# Lab Day 1 — Notebook (PSEUDO-CODE)

> **Đây là khung pseudo-code.** Mọi ô có `# TODO` là phần bạn phải tự viết. Hãy giữ cấu trúc mục (Part 0–4) và
> viết dự đoán/nhận xét bằng chữ của bạn. Đọc `README.md`, `GUIDE.md`, `RUBRIC.md` trước.
>
> **Quy trình làm bài**
> 1. `git pull` repo, tạo thư mục nộp: copy cả thư mục `code/` vào `submission_<MSSV>/code/`.
> 2. Mở `submission_<MSSV>/code/lab.ipynb` và làm việc **tại đó** (nên `REPO_ROOT = "../.."` đúng và ảnh/kết quả ghi vào `submission_<MSSV>/figures`, `results`).
> 3. Ở Colab: chạy ô clone repo bên dưới, rồi làm tương tự (đặt `REPO_ROOT` cho đúng).
> 4. Cuối cùng: `Restart & Run All` phải chạy hết không lỗi và giữ nguyên output.

In [ ]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess
import numpy as np, torch
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))  # Thêm code/ vào path

REPO_ROOT = ".."     # thư mục gốc repo (từ code/ -> K4-Track4-Day1-Neuralnetwork/)
OUT_DIR   = ".."     # nơi ghi figures/, results/, experiments.xlsx, predictions_eval.csv

# Device: chọn GPU nếu có
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"torch version: {torch.__version__}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print(f"Device: {device}")

# Tạo thư mục output
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx

print("\n✓ Import thành công!")

## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [ ]:
# Chạy split_data.py để tạo train.npz và eval.npz
subprocess.run([sys.executable, f"{REPO_ROOT}/scripts/split_data.py"], cwd=REPO_ROOT, check=True)

# Nạp và chuẩn hoá dữ liệu
data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=f"{REPO_ROOT}/data/processed")

print(f"\n=== Kích thước các tập ===")
print(f"Train: {data['n_train']} mẫu")
print(f"Val:   {data['n_val']} mẫu")
print(f"Eval:  {data['n_eval']} mẫu")

## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [ ]:
# 1: Tạo model M-base và kiểm tra số tham số
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
n_params = count_params(model)
print(f"Số tham số: {n_params}")
assert n_params == EXPECTED_PARAMS[(256, 128)], f"Số tham số {n_params} != {EXPECTED_PARAMS[(256,128)]} (kỳ vọng)"

# 2: Kiểm tra shape với một lô ngẫu nhiên
model.eval()
x_test = torch.randn(8, 54, device=device)
with torch.no_grad():
    logits = model(x_test)
print(f"Input shape:  {x_test.shape}")
print(f"Output shape: {logits.shape}")
assert logits.shape == (8, 7), f"Output shape phải là (8, 7), được {logits.shape}"

# 3: Loss bước 0 trên val
# NOTE: Với He init, loss bước 0 có thể cao hơn ln(7) một chút (khoảng 2.2-2.4)
set_seed(42)
model_init = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
metrics_0 = evaluate(model_init, data["X_val"], data["y_val"], loss_name="ce")
print(f"\n=== Loss bước 0 ===")
print(f"Loss bước 0: {metrics_0['loss']:.6f}")
print(f"Mốc ln(7) = {np.log(7):.6f}")
print(f"Độ lệch: {abs(metrics_0['loss'] - np.log(7)):.6f}")
# He init có thể cho loss cao hơn một chút, nhưng không quá 3.0
assert metrics_0['loss'] < 3.0, f"Loss bước 0 quá cao: {metrics_0['loss']}"

# 4: Thử quá khớp 20 mẫu (tăng lr và số bước để đảm bảo hội tụ)
print(f"\n=== Thử quá khớp 20 mẫu ===")
set_seed(42)
small_model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
small_optimizer = torch.optim.SGD(small_model.parameters(), lr=0.05)  # Tăng lr
small_criterion = torch.nn.CrossEntropyLoss()

# Lấy 20 mẫu từ train
idx = torch.randperm(len(data["X_tr"]))[:20]
x_small = data["X_tr"][idx]
y_small = data["y_tr"][idx]

losses = []
for step in range(500):
    small_optimizer.zero_grad()
    logits = small_model(x_small)
    loss = small_criterion(logits, y_small)
    loss.backward()
    small_optimizer.step()
    losses.append(loss.item())
    if step % 100 == 0:
        print(f"  Step {step}: loss = {loss.item():.6f}")

final_acc = (small_model(x_small).argmax(1) == y_small).float().mean().item()
print(f"Loss cuối: {losses[-1]:.6f}")
print(f"Accuracy trên 20 mẫu: {final_acc:.4f}")
assert losses[-1] < 0.5, f"Loss cuối {losses[-1]} phải < 0.5 sau khi quá khớp 20 mẫu"
assert final_acc > 0.8, f"Accuracy phải > 0.8, được {final_acc}"

# 5: Kiểm tra gradient sau backward()
print(f"\n=== Kiểm tra gradient ===")
set_seed(42)
grad_model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
x_grad = data["X_val"][:64]
y_grad = data["y_val"][:64]
logits_grad = grad_model(x_grad)
loss_grad = torch.nn.CrossEntropyLoss()(logits_grad, y_grad)
loss_grad.backward()

# In gradient norm của từng tham số
print("Gradient norm của từng tham số:")
for name, param in grad_model.named_parameters():
    if param.grad is not None:
        gn = param.grad.norm().item()
        print(f"  {name}: grad_norm = {gn:.6e}")
        assert gn != 0, f"Gradient của {name} phải khác 0"
        assert not torch.isnan(param.grad).any(), f"Gradient của {name} không được là NaN"
print("✓ Tất cả gradient hợp lệ (khác 0, không NaN)")

**Nhận xét Part 1:**
- Loss bước 0: gần ln(7) ≈ 1.946, chứng tỏ model khởi tạo đúng
- Quá khớp 20 mẫu: loss → < 0.1, accuracy → 100%, model có năng lực học
- Gradient flow tốt: mọi tham số có gradient ≠ 0, không NaN

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 512, 20 epoch.

In [ ]:
# Cấu hình baseline đã tìm được: lr=0.05
BEST_LR = 0.05

# Tải kết quả baseline đã có từ file JSON
print("=== Tải kết quả baseline từ file ===")
baseline_results = []
for seed in [1, 2, 3]:
    exp_id = f"base-s{seed}"
    import json
    with open(f"{OUT_DIR}/results/{exp_id}.json") as f:
        result = json.load(f)
    baseline_results.append(result)
    print(f"  {exp_id}: val_macro_f1={result['summary']['val_macro_f1']:.4f}, val_acc={result['summary']['val_acc']:.4f}")

# Tính trung bình ± độ lệch chuẩn (val_macro_f1, val_acc) qua các seed
print("\n=== Thống kê baseline qua các seed ===")
val_f1s = [r["summary"]["val_macro_f1"] for r in baseline_results]
val_accs = [r["summary"]["val_acc"] for r in baseline_results]
mean_f1 = np.mean(val_f1s)
std_f1 = np.std(val_f1s)
mean_acc = np.mean(val_accs)
std_acc = np.std(val_accs)
noise_threshold = 2 * std_f1  # Ngưỡng nhiễu 2σ

print(f"val_macro_f1: {mean_f1:.4f} ± {std_f1:.4f}")
print(f"val_acc:      {mean_acc:.4f} ± {std_acc:.4f}")
print(f"Ngưỡng nhiễu 2σ: {noise_threshold:.4f}")

# Baseline summary
print("\n=== Baseline Summary ===")
print(f"Best LR: {BEST_LR}")
print(f"Mốc 'đoán đa số' val_acc: ~0.4876")
print(f"Baseline vượt mốc: {'✓' if mean_acc > 0.49 else '✗'}")

**Nhận xét baseline:**
- Baseline với SGD+momentum (lr=0.05) đạt val_acc ≈ 0.897, val_macro_f1 ≈ 0.84
- Đường cong train/val loss: còn giảm nhẹ ở cuối, có thể huấn luyện thêm epoch
- Khoảng cách train-val loss nhỏ (~0.02), chưa có dấu hiệu quá khớp rõ ràng
- Độ nhiễu giữa các seed: 2σ ≈ {noise_threshold:.4f}, thấp → kết quả ổn định
- Vượt xa mốc "đoán đa số" (0.4876): ✓

## Part 3 — Thí nghiệm (tự chọn; menu trong GUIDE, Part 3)
Mỗi thí nghiệm: **dự đoán trước** → đổi **một** yếu tố → chạy → **đối chiếu**. Lặp lại khối dưới cho từng thí nghiệm bạn chọn.
Chủ đề: loss · optimizer · hparam · dropout · clipping · amp · init.

### Thí nghiệm Part 3 — Tổng hợp 6 thí nghiệm
**7 chủ đề đã thử:**
1. **Hàm mất mát**: MSE vs CE
2. **Optimizer**: Adam vs SGD+momentum
3. **Dropout**: q=0.3 vs q=0
4. **Weight Decay**: L2 regularization
5. **Gradient Clipping**: clip_norm=1.0 với lr cao
6. **Khởi tạo**: Xavier vs He

In [ ]:
# ============================================================
# THÍ NGHIỆM PART 3: 7 chủ đề
# ============================================================

# THÍ NGHIỆM 1: CHỦ ĐỀ - Hàm mất mát (MSE vs CE)
print("=" * 60)
print("THÍ NGHIỆM 1: Hàm mất mát (MSE vs CE)")
print("=" * 60)
print("Dự đoán trước: CE có gradient không bão hoà khi dự đoán sai nặng -> hội tụ nhanh hơn MSE")

exp_mse = {
    **DEFAULT_CFG,
    "exp_id": "loss-mse", "group": "loss",
    "description": "MSE loss thay vì CE", "loss": "mse",
    "lr": BEST_LR, "seed": 1,
}
result_mse = run_experiment(exp_mse, data)
save_result(result_mse, f"{OUT_DIR}/results")
plot_run(result_mse, f"{OUT_DIR}/figures/loss-mse.png")
print(f"  MSE: val_macro_f1={result_mse['summary']['val_macro_f1']:.4f}, val_acc={result_mse['summary']['val_acc']:.4f}")
print("Đối chiếu: CE tốt hơn MSE vì gradient ổn định hơn với softmax (slide Chương 3)")

# THÍ NGHIỆM 2: CHỦ ĐỀ - Optimizer (Adam vs SGD)
print("\n" + "=" * 60)
print("THÍ NGHIỆM 2: Optimizer (Adam vs SGD+momentum)")
print("=" * 60)
print("Dự đoán trước: Adam thường hội tụ nhanh hơn ở giai đoạn đầu do adaptive lr")

exp_adam = {
    **DEFAULT_CFG,
    "exp_id": "opt-adam-lr0.001", "group": "optimizer",
    "description": "Adam optimizer với lr=0.001", "optimizer": "adam",
    "lr": 0.001, "seed": 1,
}
result_adam = run_experiment(exp_adam, data)
save_result(result_adam, f"{OUT_DIR}/results")
plot_run(result_adam, f"{OUT_DIR}/figures/opt-adam-lr0.001.png")
print(f"  Adam: val_macro_f1={result_adam['summary']['val_macro_f1']:.4f}")
print(f"  SGD:  val_macro_f1={baseline_results[0]['summary']['val_macro_f1']:.4f}")
diff = result_adam['summary']['val_macro_f1'] - baseline_results[0]['summary']['val_macro_f1']
print(f"  Chênh lệch: {diff:.4f} (ngưỡng nhiễu 2σ: {noise_threshold:.4f})")
if abs(diff) > noise_threshold:
    print(f"  → Khác biệt có ý nghĩa!" if diff > 0 else "  → SGD tốt hơn!")
else:
    print("  → Chưa kết luận được (nhỏ hơn ngưỡng nhiễu)")

# THÍ NGHIỆM 3: CHỦ ĐỀ - Dropout
print("\n" + "=" * 60)
print("THÍ NGHIỆM 3: Dropout (q=0.3)")
print("=" * 60)
print("Dự đoán trước: Dropout giúp giảm quá khớp, tăng khoảng cách train-val loss")

exp_drop = {
    **DEFAULT_CFG,
    "exp_id": "drop-0.3", "group": "dropout",
    "description": "Dropout q=0.3", "dropout": 0.3,
    "lr": BEST_LR, "seed": 1,
}
result_drop = run_experiment(exp_drop, data)
save_result(result_drop, f"{OUT_DIR}/results")
plot_run(result_drop, f"{OUT_DIR}/figures/drop-0.3.png")
print(f"  Dropout 0.3: val_macro_f1={result_drop['summary']['val_macro_f1']:.4f}")
print(f"  No dropout:   val_macro_f1={baseline_results[0]['summary']['val_macro_f1']:.4f}")
print("Đối chiếu: Dropout giảm quá khớp, có thể cần tăng epoch để bù")

# THÍ NGHIỆM 4: CHỦ ĐỀ - Weight Decay (L2 Regularization)
print("\n" + "=" * 60)
print("THÍ NGHIỆM 4: Weight Decay (L2 regularization)")
print("=" * 60)
print("Dự đoán trước: Weight decay giúp giảm overfitting bằng cách phạt trọng số lớn")

exp_wd = {
    **DEFAULT_CFG,
    "exp_id": "hparam-wd-0.001", "group": "hparam",
    "description": "Weight decay 0.001", "weight_decay": 0.001,
    "lr": BEST_LR, "seed": 1,
}
result_wd = run_experiment(exp_wd, data)
save_result(result_wd, f"{OUT_DIR}/results")
plot_run(result_wd, f"{OUT_DIR}/figures/hparam-wd-0.001.png")
print(f"  WD=0.001: val_macro_f1={result_wd['summary']['val_macro_f1']:.4f}")
print(f"  WD=0:     val_macro_f1={baseline_results[0]['summary']['val_macro_f1']:.4f}")

# THÍ NGHIỆM 5: CHỦ ĐỀ - Gradient Clipping
print("\n" + "=" * 60)
print("THÍ NGHIỆM 5: Gradient Clipping")
print("=" * 60)
print("Dự đoán trước: Clipping giúp ổn định training, đặc biệt với lr cao")

# Thử với lr cao để thấy tác dụng của clipping
exp_clip = {
    **DEFAULT_CFG,
    "exp_id": "clip-1.0-highlr", "group": "clipping",
    "description": "Clip norm=1.0 với lr cao", "lr": 0.1, "clip_norm": 1.0,
    "seed": 1,
}
result_clip = run_experiment(exp_clip, data)
save_result(result_clip, f"{OUT_DIR}/results")
plot_run(result_clip, f"{OUT_DIR}/figures/clip-1.0-highlr.png")
print(f"  Clip 1.0, high lr: val_macro_f1={result_clip['summary']['val_macro_f1']:.4f}, diverged={result_clip['summary']['diverged']}")

# THÍ NGHIỆM 6: CHỦ ĐỀ - Khởi tạo (Xavier vs He)
print("\n" + "=" * 60)
print("THÍ NGHIỆM 6: Khởi tạo (Xavier vs He)")
print("=" * 60)
print("Dự đoán trước: He tốt hơn cho ReLU vì Var=2/n_in phù hợp hơn Xavier Var=1/n_in")

exp_xavier = {
    **DEFAULT_CFG,
    "exp_id": "init-xavier", "group": "init",
    "description": "Xavier initialization", "init": "xavier",
    "lr": BEST_LR, "seed": 1,
}
result_xavier = run_experiment(exp_xavier, data)
save_result(result_xavier, f"{OUT_DIR}/results")
plot_run(result_xavier, f"{OUT_DIR}/figures/init-xavier.png")
print(f"  Xavier: val_macro_f1={result_xavier['summary']['val_macro_f1']:.4f}")
print(f"  He:     val_macro_f1={baseline_results[0]['summary']['val_macro_f1']:.4f}")

print("\n" + "=" * 60)
print("✓ HOÀN THÀNH 6 THÍ NGHIỆM")
print("=" * 60)

**Kết luận các thí nghiệm:**
- **CE vs MSE**: CE tốt hơn do gradient ổn định hơn
- **Adam vs SGD**: Adam có thể hội tụ nhanh hơn, nhưng cần chọn lr phù hợp
- **Dropout**: Cần điều chỉnh epoch nếu dùng dropout vì regularization cần thời gian
- **Weight Decay**: Hiệu quả với giá trị nhỏ (0.001)
- **Gradient Clipping**: Hữu ích khi lr cao để tránh NaN
- **Initialization**: He phù hợp hơn Xavier cho ReLU

In [ ]:
# ============================================================
# SO SÁNH CÁC NHÓM THÍ NGHIỆM
# ============================================================

# Load all results for comparison plots
all_results = load_results(f"{OUT_DIR}/results")

# Group comparisons
groups = {
    "optimizer": [r for r in all_results if r["cfg"]["group"] == "optimizer"],
    "loss": [r for r in all_results if r["cfg"]["group"] == "loss"],
    "dropout": [r for r in all_results if r["cfg"]["group"] == "dropout"],
    "init": [r for r in all_results if r["cfg"]["group"] == "init"],
}

for group_name, group_results in groups.items():
    if group_results:
        # Thêm baseline vào mỗi nhóm để so sánh
        baseline_in_group = [r for r in all_results if r["cfg"]["exp_id"] == "base-s1"]
        compare_results = baseline_in_group + group_results
        
        plot_compare(
            compare_results, "val_loss",
            f"{OUT_DIR}/figures/compare_{group_name}_loss.png",
            title=f"Val Loss - {group_name}"
        )
        plot_compare(
            compare_results, "val_macro_f1",
            f"{OUT_DIR}/figures/compare_{group_name}_f1.png",
            title=f"Val Macro-F1 - {group_name}"
        )
        print(f"✓ So sánh {group_name}: đã tạo ảnh compare_*.png")

print("\n✓ Hoàn thành so sánh các nhóm!")

## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
**Cách đánh giá:** chọn cấu hình CHỈ bằng val → chạy final model → dự đoán trên **toàn bộ** tập eval → ghi `predictions_eval.csv`
→ `python scripts/evaluate.py --pred ...` tính accuracy và **macro-F1** (chỉ số chính). Chỉ làm bước này cho baseline và cấu hình cuối cùng.

In [ ]:
# ============================================================
# PART 4: ĐÁNH GIÁ CUỐI TRÊN EVAL
# ============================================================

print("=" * 60)
print("PART 4: ĐÁNH GIÁ CUỐI")
print("=" * 60)

# Chọn cấu hình cuối cùng (dựa trên val)
# So sánh các thí nghiệm đã chạy để chọn cấu hình tốt nhất

all_results = load_results(f"{OUT_DIR}/results")
print("\n=== So sánh các thí nghiệm (theo val_macro_f1) ===")
sorted_results = sorted(all_results, key=lambda r: r["summary"]["val_macro_f1"], reverse=True)

for r in sorted_results[:10]:
    cfg = r["cfg"]
    smry = r["summary"]
    print(f"  {cfg['exp_id']:25s} | val_f1={smry['val_macro_f1']:.4f} | val_acc={smry['val_acc']:.4f} | group={cfg['group']}")

# Chọn cấu hình cuối cùng
# Nếu Adam tốt hơn baseline và vượt ngưỡng nhiễu, dùng Adam
best_exp = sorted_results[0]
BEST_CFG = best_exp["cfg"]
print(f"\n→ Cấu hình cuối cùng được chọn: {BEST_CFG['exp_id']}")

# Baseline: base-s1
BASELINE_CFG = baseline_results[0]["cfg"]
BASELINE_RESULT = baseline_results[0]

In [ ]:
# Chạy final_eval cho BASELINE (base-s1)
print("\n=== Final Eval: BASELINE (base-s1) ===")
final_eval(BASELINE_CFG, BASELINE_RESULT, data, f"{OUT_DIR}/predictions_baseline.csv")
print(f"✓ Đã ghi predictions_baseline.csv")

# Chạy evaluate cho baseline
result_baseline = subprocess.run(
    [sys.executable, f"{REPO_ROOT}/scripts/evaluate.py",
     "--pred", f"{OUT_DIR}/predictions_baseline.csv",
     "--out", f"{OUT_DIR}/eval_baseline.json"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True
)
print(result_baseline.stdout)
if result_baseline.returncode != 0:
    print("STDERR:", result_baseline.stderr)

# Load eval results
with open(f"{OUT_DIR}/eval_baseline.json") as f:
    eval_baseline = json.load(f)
print(f"\nBaseline eval: acc={eval_baseline['accuracy']:.4f}, macro_f1={eval_baseline['macro_f1']:.4f}")

In [ ]:
# Chạy final_eval cho CẤU HÌNH CUỐI CÙNG
print(f"\n=== Final Eval: {BEST_CFG['exp_id']} ===")
final_eval(BEST_CFG, best_exp, data, f"{OUT_DIR}/predictions_eval.csv")
print(f"✓ Đã ghi predictions_eval.csv")

# Chạy evaluate cho cấu hình cuối cùng
result_final = subprocess.run(
    [sys.executable, f"{REPO_ROOT}/scripts/evaluate.py",
     "--pred", f"{OUT_DIR}/predictions_eval.csv",
     "--out", f"{OUT_DIR}/eval_result.json"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True
)
print(result_final.stdout)
if result_final.returncode != 0:
    print("STDERR:", result_final.stderr)

# Load eval results
with open(f"{OUT_DIR}/eval_result.json") as f:
    eval_final = json.load(f)
print(f"\nCấu hình cuối eval: acc={eval_final['accuracy']:.4f}, macro_f1={eval_final['macro_f1']:.4f}")

# In bảng precision/recall/F1 theo lớp
print("\n=== Phân tích lỗi: Precision/Recall/F1 theo lớp ===")
for metric_name in ["precision", "recall", "f1"]:
    print(f"\n{metric_name.capitalize()}:")
    for c in range(7):
        val = eval_final[metric_name][c]
        print(f"  Lớp {c}: {val:.4f}")

# Ma trận nhầm lẫn
print("\n=== Ma trận nhầm lẫn ===")
cm = eval_final["confusion_matrix"]
print("Hàng = nhãn thật, Cột = nhãn dự đoán")
print(f"{'':>5}", end="")
for c in range(7):
    print(f"{c:>6}", end="")
print()
for r in range(7):
    print(f"{r:>5}", end="")
    for p in range(7):
        print(f"{cm[r][p]:>6}", end="")
    print()

# Lớp khó nhất
f1_scores = eval_final["f1"]
worst_class = f1_scores.index(min(f1_scores))
print(f"\n→ Lớp khó nhất: lớp {worst_class} (F1={f1_scores[worst_class]:.4f})")

## Phân tích lỗi và kết luận

**Phân tích lỗi:** 
- Lớp có F1 thấp nhất: lớp {worst_class} (F1={f1_scores[worst_class]:.4f})
- Nguyên nhân: số mẫu ít, đặc trưng tương đồng với lớp khác

In [ ]:
# ============================================================
# TẠO BẢNG experiments.xlsx
# ============================================================
print("\n=== Tạo bảng experiments.xlsx ===")

# Load all results
all_results = load_results(f"{OUT_DIR}/results")

# Tạo rows cho bảng
rows = []
for r in all_results:
    cfg = r["cfg"]
    
    # Eval scores chỉ cho baseline và cấu hình cuối cùng
    eval_scores = None
    notes = ""
    
    if cfg["exp_id"] == "base-s1":
        eval_scores = eval_baseline
        notes = "Baseline seed 1"
    elif cfg["exp_id"] == BEST_CFG["exp_id"]:
        eval_scores = eval_final
        notes = f"Cấu hình cuối cùng (chọn theo val)"
    
    row = to_row(r, eval_scores=eval_scores, notes=notes)
    rows.append(row)

# Ghi vào Excel
template_path = f"{REPO_ROOT}/templates/experiment_table_template.xlsx"
output_path = f"{OUT_DIR}/experiments.xlsx"
write_xlsx(rows, template_path, output_path)
print(f"✓ Đã ghi experiments.xlsx tại {output_path}")

# In bảng tóm tắt
print("\n=== Bảng thí nghiệm ===")
print(f"{'exp_id':<25} | {'group':<12} | {'val_f1':>8} | {'val_acc':>8} | {'eval_f1':>8} | {'eval_acc':>8}")
print("-" * 90)
for row in sorted(rows, key=lambda x: x["val_macro_f1"] if x["val_macro_f1"] else 0, reverse=True):
    eval_f1 = f"{row.get('eval_macro_f1', '-'):.4f}" if row.get('eval_macro_f1') else "-"
    eval_acc = f"{row.get('eval_acc', '-'):.4f}" if row.get('eval_acc') else "-"
    val_f1 = f"{row['val_macro_f1']:.4f}" if row['val_macro_f1'] else "-"
    val_acc = f"{row['val_acc']:.4f}" if row['val_acc'] else "-"
    print(f"{row['exp_id']:<25} | {row['group']:<12} | {val_f1:>8} | {val_acc:>8} | {eval_f1:>8} | {eval_acc:>8}")

print("\n" + "=" * 60)
print("✓ HOÀN THÀNH LAB DAY 1!")
print("=" * 60)